In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_1206.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_932.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_2288.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_1416.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_1746.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_1159.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_767.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_1780.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_973.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_930.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_611.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_1179.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_2153.jpg
/kaggle/input/ue-citydata-2/SyntheticData/val/RGB/rgb_frame_970.jpg
/kaggle/input/ue-citydata-2/SyntheticDat

In [3]:
import torch.optim as optim
from tqdm import tqdm 
import matplotlib.pyplot as plt
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch

In [4]:
from glob import glob
import os

train_rgb_paths = glob("/kaggle/input/ue-citydata-2/SyntheticData/train/RGB/*.jpg")
train_mask_paths = glob("/kaggle/input/ue-citydata-2/SyntheticData/train/Semantic/*.jpg")

print("RGB images found:", len(train_rgb_paths))
print("Mask images found:", len(train_mask_paths))

RGB images found: 1655
Mask images found: 1655


In [5]:
# === class_colors with your dataset
class_colors = {
    "freeway deck": [0, 81, 33],
    "obstacles": [83, 21, 0],
    "traffic light": [0, 89, 254],
    "decor": [142, 255, 145],
    "margin": [222, 118, 255],
    "building": [255, 239, 3],
    "car": [255, 0, 86],
    "road": [0, 184, 255],
    "sky": [255, 255, 255],
}

# Convert to index mappings
CLASS_NAMES = list(class_colors.keys())
COLOR_TO_CLASS = {tuple(v): i for i, v in enumerate(class_colors.values())}
CLASS_TO_COLOR = {i: tuple(v) for i, v in enumerate(class_colors.values())}
color_to_class = {tuple(rgb): idx for idx, (class_name, rgb) in enumerate(class_colors.items())}

# Function to convert RGB mask to index mask
import numpy as np

def mask_rgb_to_class_index(batch_rgb_masks, color_to_class):
    batch_class_masks = []

    for rgb_mask in batch_rgb_masks:
        # Ensure input is (C, H, W)
        if rgb_mask.ndim == 3 and rgb_mask.shape[0] == 3:
            rgb_mask = np.transpose(rgb_mask, (1, 2, 0))  # (C, H, W) -> (H, W, C)
        elif rgb_mask.ndim == 3 and rgb_mask.shape[2] == 3:
            pass  # Already (H, W, C)
        else:
            raise ValueError(f"Unexpected rgb_mask shape: {rgb_mask.shape}")

        class_mask = np.zeros(rgb_mask.shape[:2], dtype=np.int64)

        for color, class_idx in color_to_class.items():
            color = np.array(color)
            matches = np.all(rgb_mask == color, axis=-1)
            class_mask[matches] = class_idx

        batch_class_masks.append(torch.tensor(class_mask))

    return torch.stack(batch_class_masks)

In [6]:
from torch.utils.data import Dataset
from PIL import Image
import os

class SemanticSegDataset(Dataset):
    def __init__(self, rgb_dir, mask_dir, rgb_transform=None, mask_transform=None):
        self.rgb_dir = rgb_dir
        self.mask_dir = mask_dir
        self.rgb_transform = rgb_transform
        self.mask_transform = mask_transform
        self.filenames = sorted(os.listdir(rgb_dir))

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        rgb_name = self.filenames[idx]
        rgb_path = os.path.join(self.rgb_dir, rgb_name)
        mask_name = rgb_name.replace("rgb", "semantic")
        mask_path = os.path.join(self.mask_dir, mask_name)

        rgb = Image.open(rgb_path).convert("RGB")
        mask = Image.open(mask_path)

        if self.rgb_transform:
            rgb = self.rgb_transform(rgb)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        return rgb, mask

In [7]:
from torch.cuda.amp import autocast, GradScaler

In [22]:
import torchvision.transforms as T
from torch.utils.data import DataLoader

rgb_transform = T.Compose([
    T.Resize(520),  # Resize shorter side
    T.CenterCrop((512, 512)),
    T.ToTensor(),
])

mask_transform = T.Compose([
    T.Resize(520, interpolation=Image.NEAREST),
    T.CenterCrop((512, 512)),
    T.PILToTensor(),
])

train_dataset = SemanticSegDataset("/kaggle/input/ue-citydata-2/SyntheticData/train/RGB", "/kaggle/input/ue-citydata-2/SyntheticData/train/Semantic", rgb_transform, mask_transform)
val_dataset = SemanticSegDataset("/kaggle/input/ue-citydata-2/SyntheticData/val/RGB", "/kaggle/input/ue-citydata-2/SyntheticData/val/Semantic", rgb_transform, mask_transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

In [9]:
import torchvision.models.segmentation as models
import torch.nn as nn
import torch

NUM_CLASSES = 9 

model = models.deeplabv3_resnet50(weights="DEFAULT")
model.classifier[4] = nn.Conv2d(256, NUM_CLASSES, kernel_size=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth
100%|██████████| 161M/161M [00:00<00:00, 226MB/s] 


DeepLabV3(
  (backbone): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Se

In [10]:
import tensorflow as tf

num_classes = len(class_colors)
class_names = [
    "freeway deck",  # 0
    "obstacles",     # 1
    "traffic light", # 2
    "decor",         # 3
    "margin",        # 4
    "building",      # 5
    "car",           # 6
    "road",          # 7
    "sky",           # 8
]

# Normalized weights for each class (from previous calculation)
class_weights = [
    0.0109,  # freeway deck
    1.0000,  # obstacles
    0.1643,  # traffic light
    0.0120,  # decor
    0.0142,  # margin
    0.0014,  # building
    0.0145,  # car
    0.0022,  # road
    0.0052   # sky
]

# Convert to tensor
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# Initialize CrossEntropyLoss with class weights
criterion = nn.CrossEntropyLoss(weight=weights_tensor.to(device))

2025-05-18 12:41:42.000239: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747572102.180467      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747572102.237749      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=5, verbose=True, min_lr=2e-6)

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [12]:
class_colors = {
    "freeway deck": [0, 81, 33],
    "obstacles": [83, 21, 0],
    "traffic light": [0, 89, 254], 
    "decor": [142, 255, 145],
    "margin": [222, 118, 255],
    "building": [255, 239, 3],
    "car": [255, 0, 86],
    "road": [0, 184, 255],
    "sky": [255, 255, 255],
}

color_to_class = {tuple(rgb): idx for idx, (class_name, rgb) in enumerate(class_colors.items())}

def rgb_to_class_indices(batch_rgb_masks, color_to_class):
    batch_class_masks = []
    for rgb_mask in batch_rgb_masks:
        rgb_mask = rgb_mask.transpose(1, 2, 0)  # (C, H, W) to (H, W, C)
        class_mask = np.zeros(rgb_mask.shape[:2], dtype=np.int64)
        for color, class_idx in color_to_class.items():
            mask = np.all(rgb_mask == np.array(color), axis=-1)
            class_mask[mask] = class_idx
        batch_class_masks.append(torch.tensor(class_mask, dtype=torch.long))
    return torch.stack(batch_class_masks)

In [27]:
scaler = torch.cuda.amp.GradScaler()

for images, rgb_masks in train_loader:
    images = images.to(device)

    # rgb_masks is [B, 3, H, W], convert to numpy and then class index
    masks = rgb_to_class_indices(rgb_masks.numpy(), color_to_class)  # [B, H, W]
    masks = masks.to(device)

    with torch.cuda.amp.autocast():
        outputs = model(images)['out']  # extract logits from OrderedDict
        loss = criterion(outputs, masks)


    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

/tmp/ipykernel_156/1275179678.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_156/1275179678.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


In [13]:
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience = patience
        self.counter = 0
        self.best_acc = 0
        self.best_model_wts = None

    def step(self, acc, model):
        if acc > self.best_acc:
            self.best_acc = acc
            self.best_model_wts = model.state_dict()
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_model_wts:
            model.load_state_dict(self.best_model_wts)

early_stopping = EarlyStopping(patience=10)

In [14]:
def compute_iou_dice(preds, masks, num_classes):
    ious = []
    dices = []
    preds = preds.view(-1)
    masks = masks.view(-1)
    for cls in range(num_classes):
        pred_inds = preds == cls
        target_inds = masks == cls
        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()
        dice = (2 * intersection) / (pred_inds.sum() + target_inds.sum() + 1e-6)
        iou = intersection / (union + 1e-6)
        if target_inds.sum() > 0: 
            ious.append(iou.item())
            dices.append(dice.item())
    return sum(ious) / len(ious), sum(dices) / len(dices)

In [ ]:
EPOCHS = 10
epoch_losses, epoch_accuracies, val_losses, val_accuracies = [], [], [], []
iou_scores, dice_scores = [], []
# scaler = GradScaler()



for epoch in range(EPOCHS):
    # -------- Training --------
    model.train()
    total_loss = 0
    total_correct = 0
    total_pixels = 0

    for images, rgb_masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - Training", unit="batch"):
        images = images.to(device)
        masks = rgb_to_class_indices(rgb_masks.numpy(), color_to_class).to(device)

        outputs = model(images)['out']
        loss = criterion(outputs, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        total_correct += (preds == masks).sum().item()
        total_pixels += masks.numel()

    avg_train_loss = total_loss / len(train_loader)
    train_accuracy = total_correct / total_pixels * 100
    epoch_losses.append(avg_train_loss)
    epoch_accuracies.append(train_accuracy)

    # -------- Validation --------
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    total_iou = 0
    total_dice = 0
    num_batches = 0

    with torch.no_grad():
        for images, rgb_masks in tqdm(val_loader, desc="Validating", unit="batch"):
            images = images.to(device)
            masks = rgb_to_class_indices(rgb_masks.numpy(), color_to_class).to(device)

            outputs = model(images)['out']
            loss = criterion(outputs, masks)
            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == masks).sum().item()
            val_total += masks.numel()

            # Compute IoU and Dice
            iou, dice = compute_iou_dice(preds, masks, NUM_CLASSES)
            total_iou += iou
            total_dice += dice
            num_batches += 1

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = val_correct / val_total * 100
    avg_iou = total_iou / num_batches
    avg_dice = total_dice / num_batches

    val_losses.append(avg_val_loss)
    val_accuracies.append(val_accuracy)
    iou_scores.append(avg_iou)
    dice_scores.append(avg_dice)

    # Scheduler and Early Stopping
    scheduler.step(val_accuracy)
    stop = early_stopping.step(val_accuracy, model)

    print(f"Epoch {epoch+1}/{EPOCHS}, "
          f"Loss: {avg_train_loss:.4f}, Acc: {train_accuracy:.2f}%, "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%, "
          f"IoU: {avg_iou:.4f}, Dice: {avg_dice:.4f}")

    if stop:
        print("Early stopping triggered.")
        break

early_stopping.restore(model)
plt.figure(figsize=(21, 6))

plt.subplot(1, 3, 1)
plt.plot(range(1, len(epoch_losses)+1), epoch_losses, label='Train Loss', color='red')
plt.plot(range(1, len(val_losses)+1), val_losses, label='Val Loss', color='orange')
plt.title("Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(range(1, len(epoch_accuracies)+1), epoch_accuracies, label='Train Accuracy', color='blue')
plt.plot(range(1, len(val_accuracies)+1), val_accuracies, label='Val Accuracy', color='green')
plt.title("Accuracy over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.grid(True)
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(range(1, len(iou_scores)+1), iou_scores, label='IoU', color='purple')
plt.plot(range(1, len(dice_scores)+1), dice_scores, label='Dice', color='brown')
plt.title("IoU and Dice over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
import torch
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

model.eval()

rgb_path = "/kaggle/input/ue-citydata-2/SyntheticData/test/RGB/rgb_frame_1162.jpg"               # ← Update
gt_mask_path = "/kaggle/input/ue-citydata-2/SyntheticData/test/Semantic/semantic_frame_1162.jpg" # ← Update

# Load RGB and GT mask
image = Image.open(rgb_path).convert("RGB")
gt_mask = Image.open(gt_mask_path).convert("RGB")
original_size = image.size

# Transform
transform = transforms.Compose([
    transforms.Resize((1240, 1240)),
    transforms.ToTensor()
])

# Process input image
input_tensor = transform(image).unsqueeze(0).to(device)

# Inference
model.eval()
with torch.no_grad():
    output = model(input_tensor)['out']
    pred = torch.argmax(output.squeeze(), dim=0).cpu().numpy()

# Decode prediction
def decode_segmentation(mask, class_colors):
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for idx, (class_name, color) in enumerate(class_colors.items()):
        color_mask[mask == idx] = color
    return color_mask

pred_color = decode_segmentation(pred, class_colors)
pred_image_resized = Image.fromarray(pred_color).resize(original_size)

# Plot
plt.figure(figsize=(18, 6))

# Original RGB
plt.subplot(1, 3, 1)
plt.title("Original RGB")
plt.imshow(image)
plt.axis("off")

# Ground Truth
plt.subplot(1, 3, 2)
plt.title("Ground Truth")
plt.imshow(gt_mask)
plt.axis("off")

# Predicted Segmentation
plt.subplot(1, 3, 3)
plt.title("Predicted Segmentation")
plt.imshow(pred_image_resized)
plt.axis("off")

plt.tight_layout()
plt.show()

# U-Net

In [ ]:
import torch

In [ ]:
pip install segmentation-models-pytorch

In [ ]:
CLASS_COLORS = {
    (0, 81, 33): 0,       # freeway deck
    (83, 21, 0): 1,       # obstacles
    (0, 89, 254): 2,      # traffic light
    (142, 255, 145): 3,   # decor
    (222, 118, 255): 4,   # margin
    (255, 239, 3): 5,     # building
    (255, 0, 86): 6,      # car
    (0, 184, 255): 7,     # road
    (255, 255, 255): 8,   # sky
}

def rgb_to_class(mask):
    mask = np.array(mask)
    class_mask = np.zeros((mask.shape[0], mask.shape[1]), dtype=np.uint8)

    for rgb, class_idx in CLASS_COLORS.items():
        matches = np.all(mask == rgb, axis=-1)
        class_mask[matches] = class_idx

    return class_mask

In [ ]:
class SemanticSegDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_transform=None, mask_transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_transform = image_transform
        self.mask_transform = mask_transform
        self.image_names = sorted(os.listdir(image_dir))  # Ex: rgb_frame_2256.jpg

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]  # e.g., rgb_frame_2256.jpg
        image_path = os.path.join(self.image_dir, image_name)

        # Replace "rgb_" with "semantic_" for the mask
        mask_name = image_name.replace("rgb_", "semantic_")
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path)

        if self.image_transform:
            image = self.image_transform(image)

        mask = rgb_to_class(mask)  # shape: (H, W)
        mask = Image.fromarray(mask)
        if self.mask_transform:
            mask = self.mask_transform(mask)  # mask is now a tensor
        mask = mask.squeeze(0).long() 
        # if self.image_transform:
        #     image = self.image_transform(image)
        # if self.mask_transform:
        #     mask = self.mask_transform(mask)
        #     mask = mask.squeeze(0).long()  # Convert [1, H, W] to [H, W]
        # print("Mask shape:", mask.shape, "Mask dtype:", mask.dtype)
        
        return image, mask


In [ ]:
rgb_transform = T.Compose([
    T.Resize(520),
    T.CenterCrop((512, 512)),
    T.ToTensor(),
])

mask_transform = T.Compose([
    T.Resize(520, interpolation=Image.NEAREST),
    T.CenterCrop((512, 512)),
    T.Grayscale(num_output_channels=1), 
    T.PILToTensor(),  # Output shape: [1, H, W]
])

# ----------------------------
# 3. Datasets and Loaders
# ----------------------------
train_dataset = SemanticSegDataset(
    "/kaggle/input/ue-citydata-2/SyntheticData/train/RGB",
    "/kaggle/input/ue-citydata-2/SyntheticData/train/Semantic",
    rgb_train_transform, mask_train_transform
)
val_dataset = SemanticSegDataset(
    "/kaggle/input/ue-citydata-2/SyntheticData/val/RGB",
    "/kaggle/input/ue-citydata-2/SyntheticData/val/Semantic",
    rgb_val_transform, mask_val_transform
)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

In [ ]:
import segmentation_models_pytorch as smp

# Example for multi-class segmentation (e.g. 3+ classes)
model = smp.Unet(
    encoder_name="resnet34",     # backbone
    encoder_weights="imagenet",  # pretrained on ImageNet
    in_channels=3,               # RGB
    classes=9          # number of classes in your mask
)

In [ ]:
import torch.nn as nn
from segmentation_models_pytorch.losses import DiceLoss

In [ ]:
NUM_CLASSES = 9
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="resnet34",        
    encoder_weights="imagenet",     
    in_channels=3,                  
    classes=NUM_CLASSES             
).to(device)

# ----------------------------
# 5. Loss and Optimizer
# ----------------------------
ce_loss = nn.CrossEntropyLoss()
dice_loss = DiceLoss(mode="multiclass")

def combined_loss(pred, target):
    return ce_loss(pred, target) + dice_loss(pred, target)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss
import numpy as np

In [ ]:
from tqdm import tqdm

EPOCHS = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]", leave=False)
    for images, masks in loop:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

In [ ]:


# ----------------------------
# 7. Visualization Utils
# ----------------------------
def decode_segmap(mask, num_classes):
    colors = plt.get_cmap("tab20").colors
    rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
    for c in range(num_classes):
        rgb[mask == c] = (np.array(colors[c % len(colors)]) * 255).astype(np.uint8)
    return rgb
def visualize_sample(image, gt_mask, pred_mask, num_classes):
    image_np = image.permute(1, 2, 0).cpu().numpy()
    image_np = (image_np * 255).astype(np.uint8)

    gt_rgb = decode_segmap(gt_mask.cpu().numpy(), num_classes)
    pred_rgb = decode_segmap(pred_mask.cpu().numpy(), num_classes)

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(image_np)
    axs[0].set_title("Input Image")
    axs[1].imshow(gt_rgb)
    axs[1].set_title("Ground Truth")
    axs[2].imshow(pred_rgb)
    axs[2].set_title("Prediction")

    for ax in axs:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

model.eval()
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        # Visualize the first sample
        visualize_sample(images[0], masks[0], preds[0], NUM_CLASSES)
        break  # Remove break to visualize more samples

In [ ]:
import matplotlib.pyplot as plt

# Define a color map for segmentation (adjust based on number of classes)
def decode_segmap(mask, num_classes):
    colors = plt.get_cmap("tab20").colors  # up to 20 unique colors
    rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

    for c in range(num_classes):
        rgb[mask == c] = (np.array(colors[c]) * 255).astype(np.uint8)
    return rgb

def visualize_sample(image, gt_mask, pred_mask, num_classes):
    image_np = image.permute(1, 2, 0).cpu().numpy()  # [3, H, W] -> [H, W, 3]
    image_np = (image_np * 255).astype(np.uint8)

    gt_rgb = decode_segmap(gt_mask.cpu().numpy(), num_classes)
    pred_rgb = decode_segmap(pred_mask.cpu().numpy(), num_classes)

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(image_np)
    axs[0].set_title("Input Image")
    axs[1].imshow(gt_rgb)
    axs[1].set_title("Ground Truth")
    axs[2].imshow(pred_rgb)
    axs[2].set_title("Prediction")

    for ax in axs:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
model.eval()
count = 0
with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        batch_size = images.size(0)
        for i in range(batch_size):
            visualize_sample(images[i], masks[i], preds[i], NUM_CLASSES)
            count += 1
            if count >= 70:
                break
        if count >= 70:
            break

In [ ]:
def compute_pixel_accuracy(preds, masks):
    correct = (preds == masks).sum().item()
    total = masks.numel()
    return correct, total

model.eval()
total_correct = 0
total_pixels = 0

with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        
        correct, total = compute_pixel_accuracy(preds, masks)
        total_correct += correct
        total_pixels += total

pixel_accuracy = total_correct / total_pixels
print(f"Pixel Accuracy on test set: {pixel_accuracy:.4f} ({pixel_accuracy * 100:.2f}%)")

In [ ]:
from tqdm import tqdm

EPOCHS = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]", leave=False)
    for images, masks in loop:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")